In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [2]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y_nand = np.array([1, 1, 1, 0])
y_nor  = np.array([1, 0, 0, 0])
y_xnor = np.array([1, 0, 0, 1])

In [3]:
def testar_modelo(nome, modelo, X, y):
    modelo.fit(X, y)
    pred = modelo.predict(X)

    tabela = pd.DataFrame({
        "x1": X[:, 0],
        "x2": X[:, 1],
        "esperado": y,
        "previsto": pred
    })

    print(f"\n=== {nome} ===")
    print(tabela)
    print(f"Acurácia: {accuracy_score(y, pred):.2f}")

    return modelo

In [4]:
perceptron_nand = Perceptron(
    max_iter=20,
    eta0=0.1,
    tol=None,
    shuffle=False,
    random_state=42
)

testar_modelo("Perceptron - Porta NAND", perceptron_nand, X, y_nand)


=== Perceptron - Porta NAND ===
   x1  x2  esperado  previsto
0   0   0         1         1
1   0   1         1         1
2   1   0         1         1
3   1   1         0         0
Acurácia: 1.00


Perceptron(eta0=0.1, max_iter=20, random_state=42, shuffle=False, tol=None)

In [5]:
perceptron_nor = Perceptron(
    max_iter=20,
    eta0=0.1,
    tol=None,
    shuffle=False,
    random_state=42
)

testar_modelo("Perceptron - Porta NOR", perceptron_nor, X, y_nor)


=== Perceptron - Porta NOR ===
   x1  x2  esperado  previsto
0   0   0         1         1
1   0   1         0         0
2   1   0         0         0
3   1   1         0         0
Acurácia: 1.00


Perceptron(eta0=0.1, max_iter=20, random_state=42, shuffle=False, tol=None)

In [6]:
perceptron_xnor = Perceptron(
    max_iter=20,
    eta0=0.1,
    tol=None,
    shuffle=False,
    random_state=42
)

testar_modelo("Perceptron - Porta XNOR", perceptron_xnor, X, y_xnor)


=== Perceptron - Porta XNOR ===
   x1  x2  esperado  previsto
0   0   0         1         0
1   0   1         0         0
2   1   0         0         0
3   1   1         1         0
Acurácia: 0.50


Perceptron(eta0=0.1, max_iter=20, random_state=42, shuffle=False, tol=None)

In [7]:
mlp_xnor = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation="logistic",
    solver="lbfgs",
    max_iter=10000,
    random_state=42
)

testar_modelo("MLP - Porta XNOR", mlp_xnor, X, y_xnor)


=== MLP - Porta XNOR ===
   x1  x2  esperado  previsto
0   0   0         1         1
1   0   1         0         0
2   1   0         0         0
3   1   1         1         1
Acurácia: 1.00


MLPClassifier(activation='logistic', hidden_layer_sizes=(4,), max_iter=10000,
              random_state=42, solver='lbfgs')

In [8]:
modelos = [
    ("Perceptron NAND", perceptron_nand, y_nand),
    ("Perceptron NOR", perceptron_nor, y_nor),
    ("Perceptron XNOR", perceptron_xnor, y_xnor),
    ("MLP XNOR", mlp_xnor, y_xnor)
]

resultados = []

for nome, modelo, y in modelos:
    pred = modelo.predict(X)
    acc = accuracy_score(y, pred)
    resultados.append([nome, acc])

pd.DataFrame(resultados, columns=["Modelo", "Acurácia"])

,Modelo,Acurácia
0,Perceptron NAND,1.0
1,Perceptron NOR,1.0
2,Perceptron XNOR,0.5
3,MLP XNOR,1.0


In [9]:
print("=== Arquitetura da rede ===")
print("Número de camadas:", mlp_xnor.n_layers_)
print("Número de entradas:", mlp_xnor.n_features_in_)
print("Camadas ocultas:", mlp_xnor.hidden_layer_sizes)
print("Número de saídas:", mlp_xnor.n_outputs_)

print("\n=== Pesos e bias por camada ===")

for i, (W, b) in enumerate(zip(mlp_xnor.coefs_, mlp_xnor.intercepts_)):
    print(f"\nCamada {i} -> {i+1}")
    print(f"Formato da matriz de pesos: {W.shape}")
    print(W)
    print(f"Formato do vetor de bias: {b.shape}")
    print(b)

=== Arquitetura da rede ===
Número de camadas: 3
Número de entradas: 2
Camadas ocultas: (4,)
Número de saídas: 1

=== Pesos e bias por camada ===

Camada 0 -> 1
Formato da matriz de pesos: (2, 4)
[[-6.67348625 -0.42530072  5.77638288  3.00601755]
 [ 5.87555242 -0.2705672  -6.49333515  2.8102986 ]]
Formato do vetor de bias: (4,)
[ -3.04328331 -11.16261056  -3.11139398  -0.69589127]

Camada 1 -> 2
Formato da matriz de pesos: (4, 1)
[[-14.12191792]
 [  0.09686007]
 [-14.2700716 ]
 [ -2.36976993]]
Formato do vetor de bias: (1,)
[9.036855]


In [10]:
probs = mlp_xnor.predict_proba(X)

print("x1 x2 | classe esperada | classe prevista | probabilidades")
print("----------------------------------------------------------")

for entrada, esperado, previsto, prob in zip(X, y_xnor, mlp_xnor.predict(X), probs):
    print(f"{entrada[0]}  {entrada[1]}  |       {esperado}        |        {previsto}       | {prob}")

x1 x2 | classe esperada | classe prevista | probabilidades
----------------------------------------------------------
0  0  |       1        |        1       | [9.13566119e-04 9.99086434e-01]
0  1  |       0        |        0       | [0.99836636 0.00163364]
1  0  |       0        |        0       | [0.99843968 0.00156032]
1  1  |       1        |        1       | [0.00228145 0.99771855]
